# LoFi Studio — LaMa x Fallin · Một lần upload, một file MP4

**Workflow:** Upload video animation gốc → chọn vùng logo/watermark trên frame đầu → xem thử → **xóa watermark bằng LaMa ONNX** → **upscale bằng Fallin Soft hoặc Strong FP16 trên GPU T4** → **ghi MP4 duy nhất**, giữ audio gốc.

- Không phải tải video sau khi xóa watermark về máy rồi upload lại để upscale.
- Không tạo file video trung gian nén lại; mỗi frame được xử lý trong RAM: **LaMa → Fallin → FFmpeg**.
- Mặc định **Fallin Soft → 1440p**, 24 FPS nếu nguồn là 24 FPS. Có thể chọn Strong/4K khi muốn.
- Vùng chọn giữ nguyên vị trí xuyên suốt video; logo di chuyển cần cách khác.
- Bước **Xem thử** giúp đánh giá vùng xóa và tốc độ trước khi chạy hết 10 giây.
- Không nội suy FPS. Mặc định chỉ xử lý 10 giây đầu, có thể chỉnh.

**Yêu cầu:** Colab > Runtime > Change runtime type > T4 GPU. Dùng trên nội dung bạn có quyền chỉnh sửa. Kết quả phụ thuộc video và model; notebook chưa được chạy end-to-end trên T4 trong môi trường tạo file này.

**Nguồn / giấy phép:** [LaMa-ONNX (Apache-2.0)](https://huggingface.co/Carve/LaMa-ONNX) · [Fallin Soft/Strong (CC BY 4.0, cần ghi công)](https://renarchi.github.io/Re-SISR/index.html) · [gradio-image-annotation](https://pypi.org/project/gradio-image-annotation/).

In [ ]:
# CELL 1 — Cài thư viện, dùng trực tiếp CUDA/PyTorch của Colab
import sys, subprocess

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'gradio>=5.44,<6', 'gradio-image-annotation==0.5.6',
    'onnxruntime-gpu>=1.21,<1.25', 'requests'
], check=True)

import torch, onnxruntime as ort, cv2, gradio as gr
import numpy as np
from gradio_image_annotation import image_annotator

if not torch.cuda.is_available():
    raise RuntimeError('Hãy bật GPU T4: Runtime → Change runtime type → T4 GPU.')
try:
    ort.preload_dlls()
except (AttributeError, RuntimeError) as exc:
    print('Ghi chú nạp DLL CUDA:', exc)
if 'CUDAExecutionProvider' not in ort.get_available_providers():
    raise RuntimeError('ONNX Runtime không thấy CUDAExecutionProvider; dừng để tránh xử lý chậm trên CPU.')
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda, '| ORT:', ort.__version__, '| Gradio:', gr.__version__)
print('ONNX providers:', ort.get_available_providers())
subprocess.run(['ffmpeg','-version'],stdout=subprocess.DEVNULL,check=True)
subprocess.run(['ffprobe','-version'],stdout=subprocess.DEVNULL,check=True)
print('Sẵn sàng. Chạy Cell 2.')

In [ ]:
# CELL 2 — Model, xử lý LaMa + upscale; không viết MP4 trung gian
import json, time, math, re, uuid, gc, os
import requests
from pathlib import Path
from fractions import Fraction
from PIL import Image, ImageDraw

ROOT = Path('/content/lofi_t4_one_upload')
MODELS = ROOT / 'models'
OUTPUTS = ROOT / 'outputs'
for d in (ROOT, MODELS, OUTPUTS): d.mkdir(parents=True, exist_ok=True)

CATALOG = {
    'LaMa': ('https://huggingface.co/Carve/LaMa-ONNX/resolve/main/lama_fp32.onnx',
             'lama_fp32.onnx'),
    'Fallin Soft': ('https://github.com/renarchi/Re-SISR/releases/download/Fallin/2x_Fallin_soft_renarchi_fp16.onnx',
                   '2x_Fallin_soft_renarchi_fp16.onnx'),
    'Fallin Strong': ('https://github.com/renarchi/Re-SISR/releases/download/Fallin/2x_Fallin_strong_renarchi_fp16.onnx',
                     '2x_Fallin_strong_renarchi_fp16.onnx'),
}
SESSIONS = {}


def get_weight(kind):
    url, fname = CATALOG[kind]
    dst = MODELS / fname
    if dst.exists() and dst.stat().st_size > 100_000:
        return dst
    temp = dst.with_suffix(dst.suffix + '.part')
    print('Đang tải model', kind, '…', flush=True)
    try:
        with requests.get(url, stream=True, timeout=120, headers={'User-Agent':'Colab-LoFi-Studio/1.0'}) as r:
            r.raise_for_status()
            with temp.open('wb') as f:
                for chunk in r.iter_content(chunk_size=1024*1024):
                    if chunk: f.write(chunk)
        if temp.stat().st_size < 100_000:
            raise RuntimeError('Tải model không đầy đủ.')
        temp.replace(dst)
    finally:
        temp.unlink(missing_ok=True)
    return dst


def model_session(kind):
    if kind in SESSIONS:
        return SESSIONS[kind]
    weights = get_weight(kind)
    opts = ort.SessionOptions()
    opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    opts.intra_op_num_threads = 2
    sess = ort.InferenceSession(str(weights), sess_options=opts, providers=[
        ('CUDAExecutionProvider', {'device_id': '0'}), 'CPUExecutionProvider'
    ])
    # Không reset toàn bộ session sang CPU khi GPU gặp lỗi runtime.
    sess.disable_fallback()
    if 'CUDAExecutionProvider' not in sess.get_providers():
        raise RuntimeError(f'{kind}: session không dùng CUDA. Dừng.')
    SESSIONS[kind] = sess
    print(f'{kind}: {sess.get_providers()} | {[ (i.name,i.shape,i.type) for i in sess.get_inputs() ]}', flush=True)
    return sess


def video_info(path):
    raw = subprocess.check_output([
        'ffprobe','-v','error','-select_streams','v:0',
        '-show_entries', 'stream=width,height,avg_frame_rate,r_frame_rate,nb_frames,duration:stream_tags=rotate:stream_side_data=rotation',
        '-of','json', str(path)
    ], text=True)
    streams = json.loads(raw).get('streams',[])
    if not streams: raise ValueError('File không có video.')
    s = streams[0]
    w,h = int(s['width']),int(s['height'])
    fps = Fraction(s['avg_frame_rate'])
    if fps <= 0: raise ValueError('Không đọc được FPS.')
    nominal = Fraction(s.get('r_frame_rate', '0/1'))
    if nominal > 0 and abs(float(nominal/fps)-1)>0.01:
        raise ValueError('Video dường như VFR; notebook hỗ trợ CFR để đồng bộ audio.')
    rotate=int(s.get('tags',{}).get('rotate',0) or 0)
    for x in s.get('side_data_list', []): rotate=int(x.get('rotation',rotate))
    if rotate % 360: raise ValueError('Video có rotation metadata; cần chuẩn hóa hướng trước.')
    return w,h,fps


def first_frame(video_path):
    if not video_path: raise gr.Error('Upload video trước.')
    w,h,fps = video_info(video_path)
    cap = cv2.VideoCapture(str(video_path))
    ok, frame = cap.read(); cap.release()
    if not ok: raise gr.Error('Không đọc được frame đầu tiên.')
    return {'image': cv2.cvtColor(frame,cv2.COLOR_BGR2RGB), 'boxes': []}, f'Video {w}×{h} | {float(fps):.3f} FPS. Hãy kéo một hình chữ nhật vừa khít logo.'


def annotation_box(ann, width, height):
    if ann is None or not ann.get('boxes'):
        raise gr.Error('Hãy khoanh vùng watermark trên frame đầu tiên.')
    if len(ann['boxes']) > 1:
        raise gr.Error('Hãy chỉ để lại một hình chữ nhật watermark.')
    b = ann['boxes'][0]
    x1 = max(0, min(width-1, int(b['xmin'])))
    y1 = max(0, min(height-1,int(b['ymin'])))
    x2 = max(0, min(width, int(b['xmax'])))
    y2 = max(0, min(height,int(b['ymax'])))
    if x2 <= x1 or y2 <= y1: raise gr.Error('Tọa độ watermark không hợp lệ.')
    if (x2-x1)*(y2-y1) > width*height*0.25:
        raise gr.Error('Vùng chọn quá lớn (>25% khung hình). Vui lòng khoanh sát logo.')
    return x1,y1,x2,y2


def crop_settings(box, width, height):
    x1,y1,x2,y2 = box
    w,h=x2-x1,y2-y1
    margin=max(40,int(max(w,h)*0.5))
    left=max(0,x1-margin); top=max(0,y1-margin)
    right=min(width,x2+margin); bottom=min(height,y2+margin)
    mask=np.zeros((bottom-top,right-left),np.uint8)
    mask[y1-top:y2-top,x1-left:x2-left]=255
    # Pha mép nhỏ quanh logo; ngoài mask giữ nguyên pixels.
    feather=max(2,min(8,round(min(w,h)*0.04)))
    kernel=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(3,3))
    dilated=cv2.dilate(mask,kernel,iterations=1)
    sigma=max(1.0,feather/2)
    alpha=cv2.GaussianBlur(dilated.astype(np.float32)/255.0,(0,0),sigmaX=sigma)
    alpha[mask==255]=1.0
    alpha=np.clip(alpha,0,1)[...,None]
    return (left,top,right,bottom),mask,alpha


def lama_frame(frame, settings, grain=0.0):
    (left,top,right,bottom),mask,alpha=settings
    src=frame[top:bottom,left:right]
    rgb=cv2.cvtColor(src,cv2.COLOR_BGR2RGB)
    x=cv2.resize(rgb,(512,512),interpolation=cv2.INTER_LANCZOS4)
    m=cv2.resize(mask,(512,512),interpolation=cv2.INTER_NEAREST)
    ins=model_session('LaMa').get_inputs()
    names=[i.name for i in ins]
    if 'image' not in names or 'mask' not in names:
        raise RuntimeError(f'LaMa inputs lạ: {names}')
    xx=np.ascontiguousarray(x.transpose(2,0,1)[None],dtype=np.float32)/255.0
    mm=np.ascontiguousarray((m>127).astype(np.float32)[None,None])
    out=model_session('LaMa').run(None, {'image':xx,'mask':mm})[0]
    if out.shape != (1,3,512,512):
        raise RuntimeError(f'LaMa output không đúng: {out.shape}')
    # Carve LaMa trả về RGB 0..255 (khác Fallin trả về 0..1).
    recovered=np.clip(out[0].transpose(1,2,0),0,255).astype(np.uint8)
    recovered=cv2.resize(recovered,(right-left,bottom-top),interpolation=cv2.INTER_LANCZOS4)
    recovered=cv2.cvtColor(recovered,cv2.COLOR_RGB2BGR)
    if grain > 0:
        # Chỉ thêm grain ở vùng đã inpaint, không làm nhiễu nền xung quanh.
        noise=np.random.normal(0,float(grain),recovered.shape).astype(np.float32)
        recovered=np.clip(recovered.astype(np.float32)+noise,0,255).astype(np.uint8)
    patch=(recovered.astype(np.float32)*alpha + src.astype(np.float32)*(1-alpha))
    clean=frame.copy()
    clean[top:bottom,left:right]=np.clip(patch,0,255).astype(np.uint8)
    return clean


def infer_upscale(inp_bgr, model):
    s=model_session(model)
    ins=s.get_inputs()
    if len(ins)!=1 or len(ins[0].shape)!=4:
        raise RuntimeError(f'Input model {model} không hợp lệ: {[(i.name,i.shape) for i in ins]}')
    h,w=inp_bgr.shape[:2]
    pad_h=(-h)%16; pad_w=(-w)%16
    padded=cv2.copyMakeBorder(inp_bgr,0,pad_h,0,pad_w,cv2.BORDER_REFLECT_101) if pad_h or pad_w else inp_bgr
    rgb=cv2.cvtColor(padded,cv2.COLOR_BGR2RGB)
    dtype=np.float16 if ins[0].type=='tensor(float16)' else np.float32
    x=np.ascontiguousarray(rgb.transpose(2,0,1)[None],dtype=dtype)/np.array(255,dtype=dtype)
    out=s.run(None, {ins[0].name: x})[0]
    if out.ndim!=4 or out.shape[0]!=1 or out.shape[1]!=3:
        raise RuntimeError(f'Upscaler output bất thường: {out.shape}')
    arr=np.clip(out[0].transpose(1,2,0).astype(np.float32),0,1)
    bgr=cv2.cvtColor(np.round(arr*255).astype(np.uint8),cv2.COLOR_RGB2BGR)
    return bgr[:h*2,:w*2]


def upscale_tile(frame, model, core=512, halo=32):
    h,w=frame.shape[:2]
    result=np.empty((h*2,w*2,3),np.uint8)
    for y in range(0,h,core):
        for x in range(0,w,core):
            yy=min(y+core,h); xx=min(x+core,w)
            a=max(0,y-halo); b=max(0,x-halo)
            c=min(h,yy+halo); d=min(w,xx+halo)
            enlarged=infer_upscale(frame[a:c,b:d], model)
            result[y*2:yy*2,x*2:xx*2]=enlarged[(y-a)*2:(yy-a)*2,(x-b)*2:(xx-b)*2]
    return result


def full_upscale(frame, model, target_width, target_height, use_tiling=False):
    try:
        high=upscale_tile(frame,model) if use_tiling else infer_upscale(frame,model)
    except Exception as e:
        if use_tiling or not any(k in str(e).lower() for k in ('alloc','memory','out of memory','resource exhausted')):
            raise
        print('GPU hết VRAM full-frame, thử tiling 512…',flush=True)
        gc.collect()
        high=upscale_tile(frame,model)
    if high.shape[:2] != (target_height,target_width):
        interp=cv2.INTER_AREA if high.shape[0]>=target_height else cv2.INTER_CUBIC
        high=cv2.resize(high,(target_width,target_height),interpolation=interp)
    return np.ascontiguousarray(high)


def output_size(width,height,short_edge):
    short_edge=int(short_edge)
    tw=round(width*short_edge/min(width,height))
    th=round(height*short_edge/min(width,height))
    tw+=(tw%2); th+=(th%2)
    if max(tw,th)>4096: raise gr.Error('Cạnh dài output >4096 pixel. Giảm 1440p/2160p.')
    return tw,th


def nvenc_available():
    test=subprocess.run(['ffmpeg','-hide_banner','-loglevel','error','-f','lavfi','-i',
                         'color=c=black:s=64x64:r=2','-t','1','-an','-c:v','h264_nvenc',
                         '-f','null','-'], stdout=subprocess.DEVNULL,
                        stderr=subprocess.PIPE,text=True)
    return test.returncode==0

print('Đã nạp hàm xử lý. Model sẽ tải một lần ở lần xem thử đầu tiên.')

In [ ]:
# CELL 3 — Preview + export, tất cả trong một pipeline streaming
from IPython.display import display


def preview(video_path, annotation, model, short_edge, grain, tile):
    if not video_path: raise gr.Error('Bạn chưa upload video.')
    width,height,fps=video_info(video_path)
    box=annotation_box(annotation,width,height)
    tw,th=output_size(width,height,short_edge)
    cap=cv2.VideoCapture(str(video_path))
    ok,frame=cap.read(); cap.release()
    if not ok: raise gr.Error('Không lấy được frame.')
    settings=crop_settings(box,width,height)
    t0=time.perf_counter()
    clean=lama_frame(frame,settings,float(grain))
    t_lama=time.perf_counter()-t0
    up=full_upscale(clean,model,tw,th,bool(tile))
    t_all=time.perf_counter()-t0
    basic=cv2.resize(frame,(tw,th),interpolation=cv2.INTER_LANCZOS4)
    # Crop quanh watermark ở output; trái đầu vào Lanczos, phải output đã xử lý.
    x1,y1,x2,y2=box
    sx=tw/width; sy=th/height
    cx=int((x1+x2)/2*sx); cy=int((y1+y2)/2*sy)
    sz=min(480,tw,th)
    left=max(0,min(tw-sz,cx-sz//2))
    top=max(0,min(th-sz,cy-sz//2))
    a=basic[top:top+sz,left:left+sz]
    b=up[top:top+sz,left:left+sz]
    comp=np.concatenate([a,b],axis=1)
    comp=cv2.copyMakeBorder(comp,38,0,0,0,cv2.BORDER_CONSTANT,value=(35,35,35))
    cv2.putText(comp,'VIDEO GOC',(10,25),cv2.FONT_HERSHEY_SIMPLEX,0.65,(255,255,255),2)
    cv2.putText(comp,'LAMA + '+('SOFT' if model=='Fallin Soft' else 'STRONG'),(sz+10,25),cv2.FONT_HERSHEY_SIMPLEX,0.60,(255,255,255),2)
    full_preview=cv2.resize(up,(min(1152,tw),int(round(th*min(1152,tw)/tw))),interpolation=cv2.INTER_AREA)
    return (cv2.cvtColor(comp,cv2.COLOR_BGR2RGB),
            cv2.cvtColor(full_preview,cv2.COLOR_BGR2RGB),
            f'Preview OK: {tw}×{th}; LaMa {t_lama:.1f}s; tổng {t_all:.1f}s/frame (lần đầu gồm khởi tạo model).')


def export_video(video_path, annotation, model, short_edge, max_seconds, grain, tile,
                 progress=gr.Progress(track_tqdm=False)):
    if not video_path: raise gr.Error('Chưa upload video.')
    width,height,fps=video_info(video_path)
    box=annotation_box(annotation,width,height)
    tw,th=output_size(width,height,short_edge)
    settings=crop_settings(box,width,height)
    cap=cv2.VideoCapture(str(video_path))
    if not cap.isOpened(): raise gr.Error('Không thể đọc file video.')
    total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total<=0: cap.release(); raise gr.Error('Không xác định được tổng số frame.')
    frames=min(total,max(1,round(float(fps)*float(max_seconds)))) if float(max_seconds)>0 else total
    duration=frames/float(fps)
    kind=re.sub('[^a-zA-Z0-9_-]+','_',Path(video_path).stem)[:65]
    id_suffix=uuid.uuid4().hex[:6]
    dst=OUTPUTS/f'{kind}_clean_{"soft" if model=="Fallin Soft" else "strong"}_{th}p_{id_suffix}.mp4'
    log_file=OUTPUTS/f'{dst.stem}_ffmpeg.log'
    # Tải model trước khi mở ffmpeg tránh pipe bị kẹt khi tải.
    progress(0,desc='Đang chuẩn bị LaMa và Fallin trên GPU')
    model_session('LaMa'); model_session(model)
    using_nvenc=nvenc_available()
    cmd=['ffmpeg','-hide_banner','-loglevel','warning','-y',
        '-f','rawvideo','-vcodec','rawvideo','-pix_fmt','bgr24',
        '-s:v',f'{tw}x{th}','-r',str(fps),'-i','pipe:0',
        '-i',str(video_path),'-map','0:v:0','-map','1:a:0?',
        '-frames:v',str(frames),'-t',f'{duration:.9f}']
    if using_nvenc:
        cmd += ['-c:v','h264_nvenc','-preset','p4','-rc','vbr','-cq','17','-b:v','0']
    else:
        cmd += ['-c:v','libx264','-preset','veryfast','-crf','16']
    cmd += ['-pix_fmt','yuv420p','-c:a','aac','-b:a','192k','-movflags','+faststart',str(dst)]
    t0=time.perf_counter(); completed=0
    try:
        with log_file.open('wb') as log:
            proc=subprocess.Popen(cmd,stdin=subprocess.PIPE,stdout=log,stderr=subprocess.STDOUT)
            try:
                for i in range(frames):
                    ok,frame=cap.read()
                    if not ok: raise RuntimeError(f'Lỗi đọc frame {i+1}/{frames}')
                    clean=lama_frame(frame,settings,float(grain))
                    enhanced=full_upscale(clean,model,tw,th,bool(tile))
                    if proc.poll() is not None: raise RuntimeError('FFmpeg dừng trước khi nhận đủ frame.')
                    proc.stdin.write(enhanced.tobytes())
                    completed+=1
                    if completed==1 or completed%max(1,round(float(fps)))==0 or completed==frames:
                        progress(completed/frames,desc=f'LaMa → {model}: {completed}/{frames} frame')
                        elapsed=time.perf_counter()-t0
                        print(f'{completed}/{frames} | {completed/max(elapsed,0.001):.2f} frame/s | {elapsed:.1f}s',flush=True)
                proc.stdin.close()
                rc=proc.wait()
                if rc: raise RuntimeError('FFmpeg lỗi: '+log_file.read_text(errors='replace')[-1500:])
            except Exception:
                try:
                    if proc.stdin and not proc.stdin.closed: proc.stdin.close()
                except Exception: pass
                if proc.poll() is None: proc.kill(); proc.wait()
                raise
        # Kiểm tra đủ số frame, fps và độ phân giải trước khi cho tải.
        raw=subprocess.check_output(['ffprobe','-v','error','-select_streams','v:0',
             '-count_frames','-show_entries','stream=width,height,avg_frame_rate,nb_read_frames',
             '-of','json',str(dst)],text=True)
        s=json.loads(raw)['streams'][0]
        assert int(s['width'])==tw and int(s['height'])==th, 'Sai kích thước đầu ra'
        assert int(s['nb_read_frames'])==frames, 'Sai tổng frame đầu ra'
        assert Fraction(s['avg_frame_rate'])==fps, 'FPS đầu ra sai'
        elapsed=time.perf_counter()-t0
        progress(1,desc='Xong: video LaMa + Upscale đã xuất')
        report=(f'HOÀN THÀNH: {dst.name} | {tw}×{th}, {float(fps):.3f} FPS, '
                f'{frames} frame, {elapsed:.1f} giây | encoder: '
                f'{"NVIDIA NVENC" if using_nvenc else "x264 CPU"}. '
                'Đã giữ track âm thanh nếu có.')
        print(report)
        return str(dst),str(dst),report
    except Exception as exc:
        dst.unlink(missing_ok=True)
        details=log_file.read_text(errors='replace')[-2000:] if log_file.exists() else ''
        raise gr.Error(f'Đã dừng, không giữ file lỗi: {str(exc)[:900]}. FFmpeg log: {details[:900]}') from exc
    finally:
        cap.release()

print('Đã nạp preview và export. Chạy Cell 4 để mở giao diện.')

In [ ]:
# CELL 4 — Một giao diện upload → khoanh logo → xem thử → xử lý → tải MP4
with gr.Blocks(title='LoFi Studio · LaMa + Upscale T4', theme=gr.themes.Soft()) as demo:
    gr.Markdown('## 🎬 LoFi Studio · Xóa watermark + Upscale AI trong **một lần**')
    gr.Markdown('**1 upload video → 1 vùng watermark → 1 MP4 kết quả.** Bạn chỉ cần dùng video animation gốc; không cần tải/upload file trung gian.')
    with gr.Row():
        with gr.Column(scale=1):
            input_video=gr.File(label='1. Upload video animation gốc',file_types=['video'],type='filepath')
            pick=gr.Button('2. Lấy frame đầu để khoanh watermark',variant='primary')
            annotation=image_annotator(label='3. Vẽ một hình chữ nhật quanh watermark',
                                       image_type='numpy',single_box=True)
            with gr.Accordion('4. Tùy chọn upscale',open=True):
                choice=gr.Radio(['Fallin Soft','Fallin Strong'],value='Fallin Soft',label='Model AI')
                edge=gr.Dropdown([1440,2160,1080],value=1440,label='Độ phân giải cạnh ngắn đầu ra (1440 cho 2K, 2160 cho 4K)')
                max_sec=gr.Slider(1,60,value=10,step=1,label='Số giây xử lý (tối đa 60)')
                grain=gr.Slider(0,5,value=0.5,step=0.1,label='Film grain trong vùng sửa (0 = không thêm)')
                tile=gr.Checkbox(value=False,label='Chia ô 512px (dùng nếu GPU hết VRAM)')
            preview_btn=gr.Button('🔍 Xem thử 1 frame (không chạy toàn video)')
            run_btn=gr.Button('🚀 XÓA WATERMARK + UPSCALE → MP4',variant='primary')
        with gr.Column(scale=1):
            status=gr.Textbox(label='Trạng thái',interactive=False,lines=3)
            comp=gr.Image(label='So sánh vùng watermark: video gốc / video sau xử lý')
            full=gr.Image(label='Preview frame sau upscale (đã thu nhỏ để xem)')
            output_video=gr.Video(label='Video MP4 đã hoàn tất')
            output_file=gr.File(label='⬇️ Tải MP4 đã xóa watermark + upscale')
    pick.click(first_frame,inputs=[input_video],outputs=[annotation,status])
    preview_btn.click(preview,inputs=[input_video,annotation,choice,edge,grain,tile],
                      outputs=[comp,full,status],concurrency_limit=1)
    run_btn.click(export_video,inputs=[input_video,annotation,choice,edge,max_sec,grain,tile],
                  outputs=[output_video,output_file,status],concurrency_limit=1)

# Colab: share=True để mở giao diện truy cập, link Gradio là tạm thời.
demo.queue(default_concurrency_limit=1)
demo.launch(share=True,inline=True,show_error=True)

### Gợi ý sử dụng

1. Chọn **Runtime > Change runtime type > T4 GPU** rồi chạy **Cell 1 → 4**.
2. **Upload video gốc một lần.** Bấm lấy frame → khoanh sát logo → **Xem thử 1 frame** để kiểm tra hiệu ứng xóa và chi tiết upscale.
3. Nếu ổn, bấm **XÓA WATERMARK + UPSCALE → MP4**. Lúc đó hai mô hình chạy nối tiếp trên mỗi frame, chỉ xuất ra **một video**.
4. Nếu FPS nguồn là 24 thì FPS đầu ra vẫn 24. Mặc định 1080p → 1440p. Muốn giữ nét mạnh hơn, thử **Fallin Strong** trong preview trước khi chạy dài.
5. Nếu colab báo lỗi VRAM, tích **Chia ô 512px** rồi thử lại. Kết quả lưu ở `/content/lofi_t4_one_upload/outputs/` cho đến khi Colab reset.

**Giấy phép:** Fallin Soft/Strong dùng CC BY 4.0; cần ghi công tác giả trong mô tả nếu bạn dùng sản phẩm tạo ra. Ví dụ: `Upscaling: Fallin by renarchi (CC BY 4.0) — https://github.com/renarchi/Re-SISR`.

**Lưu ý kỹ thuật:** Có thể tạo rung nhẹ giữa các frame do Fallin xử lý từng frame. LaMa xóa tốt nhất khi watermark **cố định vị trí**. Không tự nhận diện watermark di chuyển.